In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from botorch.fit import fit_gpytorch_mll
from botorch.models.pairwise_gp import PairwiseGP, PairwiseLaplaceMarginalLogLikelihood
from botorch.models.transforms.input import Normalize
from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.metrics import accuracy_score, confusion_matrix


## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define dataset variable configurations
DATASET_VARS = {
    "BH_1": {
        "obj_var": "yield",
        "cat_vars": [
            "Aryl_halide_SMILES",
            "Additive_SMILES",
            "Base_SMILES",
            "Ligand_SMILES",
        ],
        "con_vars": [],
    },
    "DA": {
        "obj_var": "yield",
        "cat_vars": ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"],
        "con_vars": ["Concentration", "Temp_C"],
    },
    "alkox": {
        "obj_var": "conversion",
        "cat_vars": [],
        "con_vars": ["catalase", "peroxidase", "alcohol_oxidase", "ph"],
    },
    "oer_plate_a": {
        "obj_var": "overpotential",
        "cat_vars": [],
        "con_vars": ["ni_load", "fe_load", "co_load", "mn_load", "ce_load", "la_load"],
    },
    "p3ht": {
        "obj_var": "conductivity",
        "cat_vars": [],
        "con_vars": [
            "p3ht_content",
            "d1_content",
            "d2_content",
            "d6_content",
            "d8_content",
        ],
    },
    "photo_pce10": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "photo_wf3": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "suzuki_edbo": {
        "obj_var": "yield",
        "cat_vars": ["electrophile", "nucleophile", "base", "ligand", "solvent"],
        "con_vars": [],
    },
    "suzuki": {
        "obj_var": "yield",
        "cat_vars": [],
        "con_vars": ["temperature", "pd_mol", "arbpin", "k3po4"],
    },
}

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_ORDER = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]
MODEL_LABELS = {
    "gpt-5-mini-2025-08-07": "GPT-5 mini",
    "o4-mini-2025-04-16": "o4 mini",
    "gpt-4.1-mini-2025-04-14": "GPT-4.1 mini",
    "gpt-4o-mini-2024-07-18": "GPT-4o mini",
    "claude-sonnet-4-5-20250929": "Claude Sonnet 4.5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
}

In [ ]:
TIME_TAGS = [
    "20251105222541",
    "20251105222635",
    "20251105222736",
    "20251106220930",
    "20251106221011",
    "20251106221048",
]

In [ ]:
# Read and concatenate batch output logs
batch_output_logs_df_list = []
for time_tag in TIME_TAGS:
    batch_output_logs_df_tmp = pd.read_csv(
        Path("./results_final") / f"results_{time_tag}" / "batch_output_logs.csv"
    )
    batch_output_logs_df_tmp.insert(2, "time_tag", time_tag)
    batch_output_logs_df_list.append(batch_output_logs_df_tmp)
batch_output_logs_df = pd.concat(batch_output_logs_df_list, axis=0, ignore_index=True)

# Sort by dataset_name and model
batch_output_logs_df["dataset_name"] = pd.Categorical(
    batch_output_logs_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
)
batch_output_logs_df["model"] = pd.Categorical(
    batch_output_logs_df["model"], categories=MODEL_ORDER, ordered=True
)
batch_output_logs_df = batch_output_logs_df.sort_values(
    ["dataset_name", "model"]
).reset_index(drop=True)

# Convert to string type
batch_output_logs_df["dataset_name"] = batch_output_logs_df["dataset_name"].astype(str)
batch_output_logs_df["model"] = batch_output_logs_df["model"].astype(str)

# Create a new column containing model parameters
batch_output_logs_df["model_w_params"] = batch_output_logs_df["model"].map(MODEL_LABELS)
for idx, row in batch_output_logs_df.iterrows():
    if pd.notna(batch_output_logs_df.loc[idx, "reasoning_effort"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += (
            " " + batch_output_logs_df.loc[idx, "reasoning_effort"]
        )
    elif pd.notna(batch_output_logs_df.loc[idx, "temperature"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += " temp" + str(
            round(batch_output_logs_df.loc[idx, "temperature"], 2)
        )

# Save the summarized batch output logs
batch_output_logs_df.to_csv("results_final/batch_output_logs.csv", index=False)

## Functions for preference learning

In [ ]:
def make_X(exp_df, dataset_name):
    cat_vars = DATASET_VARS[dataset_name]["cat_vars"]
    con_vars = DATASET_VARS[dataset_name]["con_vars"]

    if len(cat_vars) > 0 and len(con_vars) > 0:
        X_df = pd.concat(
            [
                pd.get_dummies(exp_df[c], prefix=c, drop_first=True, dtype=float)
                for c in cat_vars
            ],
            axis=1,
        )
        X_df = pd.concat([X_df, exp_df[con_vars]], axis=1)
    elif len(cat_vars) > 0 and len(con_vars) == 0:
        X_df = pd.concat(
            [
                pd.get_dummies(exp_df[c], prefix=c, drop_first=True, dtype=float)
                for c in cat_vars
            ],
            axis=1,
        )
    elif len(cat_vars) == 0 and len(con_vars) > 0:
        X_df = exp_df[con_vars]
    else:
        raise ValueError("Both categorical and continuous variables cannot be empty.")

    X_torch = torch.tensor(X_df.to_numpy(), dtype=torch.double)

    return X_torch


def make_comps(exp_df, pair_df):
    id2idx = {id: i for i, id in enumerate(exp_df["ID"])}

    winners, losers = [], []
    for _, row in pair_df.iterrows():
        a, b = id2idx[row["ID_A"]], id2idx[row["ID_B"]]
        if row["setup_predicted"] == "A":
            winners.append(a)
            losers.append(b)
        elif row["setup_predicted"] == "B":
            winners.append(b)
            losers.append(a)

    comps = torch.tensor(np.vstack([winners, losers]).T, dtype=torch.long)

    return comps


def train(exp_df, pair_df, dataset_name):
    X = make_X(exp_df, dataset_name)
    comps = make_comps(exp_df, pair_df)

    model = PairwiseGP(
        X,
        comps,
        input_transform=Normalize(d=X.shape[-1]),
        consolidate_rtol=0.0,
        consolidate_atol=0.0,
    )
    mll = PairwiseLaplaceMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)

    # If fitting succeeded, then mll will be in evaluation mode, i.e. mll.training == False.
    assert not mll.training

    # If you use torch.no_grad(), detach is not necessary.
    with torch.no_grad():
        utility = model.posterior(X).mean.squeeze(-1).detach().cpu().numpy()

    result_df = exp_df.copy()
    result_df["utility"] = utility

    return result_df

## Pairwise comparison, preference learning

In [ ]:
# Accuracy, confusion matrix, preference learning
scores = []
cms = {}
exp_extracted_result_dfs = {}
exp_original_result_dfs = {}

for i, batch_output_logs_df_row in batch_output_logs_df[
    ["dataset_name", "model", "model_w_params", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    time_tag = batch_output_logs_df_row.time_tag

    print(f"dataset: {dataset_name}, model {model_w_params}")

    # Read experimental data
    exp_extracted_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_extracted.csv"
    )
    exp_original_df = pd.read_csv(f"./dataset/dataset_{dataset_name}.csv")

    # Read original pair data
    pair_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_pair.csv"
    )
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    # Judge only by numerical magnitude
    pair_original_df["setup_true"] = ""
    pair_original_df.loc[
        pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
        "setup_true",
    ] = "A"
    pair_original_df.loc[
        pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
        "setup_true",
    ] = "B"
    assert len(pair_original_df.loc[pair_original_df["setup_true"] == ""]) == 0

    # Read pair prediction results
    pair_result_df = pd.read_csv(
        Path("./results_final")
        / f"results_{time_tag}"
        / f"dataset_{dataset_name}_{model}_pair_result.csv"
    )
    pair_result_df = pair_result_df.rename(columns={"setup": "setup_predicted"})

    # For datasets where lower values are better, swap A and B to align numerical order
    if dataset_name in [
        "oer_plate_a",
        "photo_pce10",
        "photo_wf3",
    ]:
        pair_result_df["setup_predicted"] = pair_result_df["setup_predicted"].replace(
            {"A": "B", "B": "A"}
        )

    # Merge results
    pair_result_df = pd.merge(
        pair_result_df, pair_original_df, on=["ID_A", "ID_B"], how="left"
    )

    # Calculate accuracy
    accuracy = accuracy_score(
        pair_result_df["setup_true"], pair_result_df["setup_predicted"]
    )

    # Create confusion matrix
    labels = ["A", "B"]
    cm = confusion_matrix(
        pair_result_df["setup_true"], pair_result_df["setup_predicted"], labels=labels
    )
    cms[(dataset_name, model_w_params, time_tag)] = cm

    # Preference learning
    exp_extracted_result_df = train(exp_extracted_df, pair_result_df, dataset_name)
    exp_extracted_result_dfs[(dataset_name, model_w_params, time_tag)] = (
        exp_extracted_result_df
    )

    exp_original_result_df = train(exp_original_df, pair_result_df, dataset_name)
    exp_original_result_dfs[(dataset_name, model_w_params, time_tag)] = (
        exp_original_result_df
    )

    # Calculate correlation coefficients
    score_dict = {
        "dataset_name": dataset_name,
        "model": model,
        "model_w_params": model_w_params,
        "time_tag": time_tag,
        "accuracy": accuracy,
    }

    for t, res_df in zip(
        ["extracted", "original"], [exp_extracted_result_df, exp_original_result_df]
    ):
        # for t, res_df in zip(["extracted"], [exp_extracted_result_df]):
        utility = res_df["utility"].to_numpy()
        y_gt = res_df[obj_var].to_numpy()
        pearson_corr = pearsonr(utility, y_gt).statistic
        pearson_corr_pvalue = pearsonr(utility, y_gt).pvalue
        spearman_corr = spearmanr(utility, y_gt).statistic
        spearman_corr_pvalue = spearmanr(utility, y_gt).pvalue
        kt_corr = kendalltau(utility, y_gt).statistic
        kt_corr_pvalue = kendalltau(utility, y_gt).pvalue

        score_dict.update(
            {
                f"pearson_corr_{t}": pearson_corr,
                f"pearson_corr_pvalue_{t}": pearson_corr_pvalue,
                f"spearman_corr_{t}": spearman_corr,
                f"spearman_corr_pvalue_{t}": spearman_corr_pvalue,
                f"kt_corr_{t}": kt_corr,
                f"kt_corr_pvalue_{t}": kt_corr_pvalue,
            }
        )

    scores.append(score_dict)

scores_df = pd.DataFrame(scores)

# Save scores
scores_df.to_csv("results_final/scores.csv", index=False)

In [ ]:
# Plot accuracy bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(data=scores_df, x="dataset_name", y="accuracy", hue="model_w_params")
plt.grid(axis="y", linestyle="--", alpha=0.3)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
plt.ylim(0.0, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(0, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig(
    "images/pairwise_comparison_accuracy.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/pairwise_comparison_accuracy.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Plot Pearson's correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=scores_df, x="dataset_name", y="pearson_corr_original", hue="model_w_params"
)
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Pearson's \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig(
    "images/preference_learning_pearson.png", format="png", dpi=600, bbox_inches="tight"
)
plt.savefig("images/preference_learning_pearson.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/preference_learning_pearson.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot Spearman's rank correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=scores_df, x="dataset_name", y="spearman_corr_original", hue="model_w_params"
)
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Spearman's rank \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig(
    "images/preference_learning_spearman.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/preference_learning_spearman.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/preference_learning_spearman.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Plot Kendall's rank correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=scores_df, x="dataset_name", y="kt_corr_original", hue="model_w_params"
)
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Kendall's rank \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig(
    "images/preference_learning_kendall.png", format="png", dpi=600, bbox_inches="tight"
)
plt.savefig("images/preference_learning_kendall.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/preference_learning_kendall.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot accuracy and Spearman's rank correlation coefficient bar charts
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax1 = axes[0]
sns.barplot(
    data=scores_df, x="dataset_name", y="accuracy", hue="model_w_params", ax=ax1
)
ax1.grid(axis="y", linestyle="--", alpha=0.3)
ax1.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax1.set_ylim(0.0, 1.0)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set_xlabel("")
ax1.set_ylabel("Accuracy", fontsize=12)
ax1.tick_params(axis="x", labelsize=10, rotation=90)

ax2 = axes[1]
sns.barplot(
    data=scores_df,
    x="dataset_name",
    y="spearman_corr_original",
    hue="model_w_params",
    ax=ax2,
)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.grid(axis="y", linestyle="--", alpha=0.3)
ax2.set_ylim(-0.4, 1.0)
ax2.set_yticks(np.arange(-0.4, 1.1, 0.1))
ax2.set_xlabel("")
ax2.set_ylabel("Spearman's rank\ncorrelation coefficient", fontsize=12)
ax2.tick_params(axis="x", labelsize=10, rotation=90)

handles, labels = ax1.get_legend_handles_labels()
ax1.get_legend().remove()
ax2.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.2),
    fontsize=10,
    frameon=True,
    ncol=2,
)

plt.tight_layout()
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_horizontal.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot accuracy and Spearman's rank correlation coefficient bar charts
fig, axes = plt.subplots(2, 1, figsize=(5, 6), sharex=True)

ax1 = axes[0]
sns.barplot(
    data=scores_df, x="dataset_name", y="accuracy", hue="model_w_params", ax=ax1
)
ax1.grid(axis="y", linestyle="--", alpha=0.3)
ax1.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax1.set_ylim(0.0, 1.0)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set_xlabel("")
ax1.set_ylabel("Accuracy", fontsize=12)
ax1.tick_params(axis="x", labelbottom=False)

ax2 = axes[1]
sns.barplot(
    data=scores_df,
    x="dataset_name",
    y="spearman_corr_original",
    hue="model_w_params",
    ax=ax2,
)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.grid(axis="y", linestyle="--", alpha=0.3)
ax2.set_ylim(-0.4, 1.0)
ax2.set_yticks(np.arange(-0.4, 1.1, 0.1))
ax2.set_xlabel("")
ax2.set_ylabel("Spearman's rank\ncorrelation coefficient", fontsize=12)
ax2.tick_params(axis="x", labelsize=10, rotation=90)

handles, labels = ax1.get_legend_handles_labels()
ax1.get_legend().remove()
ax2.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.513, 1.13),
    fontsize=10,
    frameon=True,
    ncol=2,
)

plt.tight_layout()
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot accuracy and Spearman's rank correlation coefficient bar charts
fig, axes = plt.subplots(2, 1, figsize=(5, 6), sharex=True)

ax1 = axes[0]
sns.barplot(
    data=scores_df, x="dataset_name", y="accuracy", hue="model_w_params", ax=ax1
)
ax1.grid(axis="y", linestyle="--", alpha=0.3)
ax1.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax1.set_ylim(0.0, 1.0)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set_xlabel("")
ax1.set_ylabel("Accuracy", fontsize=12)
ax1.tick_params(axis="x", labelbottom=False)

ax2 = axes[1]
sns.barplot(
    data=scores_df,
    x="dataset_name",
    y="spearman_corr_original",
    hue="model_w_params",
    ax=ax2,
)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.grid(axis="y", linestyle="--", alpha=0.3)
ax2.set_ylim(-0.4, 1.0)
ax2.set_yticks(np.arange(-0.4, 1.1, 0.1))
ax2.set_xlabel("")
ax2.set_ylabel("Spearman's rank\ncorrelation coefficient", fontsize=12)
ax2.tick_params(axis="x", labelsize=10, rotation=90)

handles, labels = ax1.get_legend_handles_labels()
ax1.get_legend().remove()
ax2.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.575),
    fontsize=10,
    frameon=True,
    ncol=1,
)

plt.tight_layout()
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical_2.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical_2.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_accuracy_preference_learning_spearman_vertical_2.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot preference learning results for original data
ncol = len(
    np.unique([model_w_params for _, model_w_params, _ in exp_original_result_dfs])
)
nsub = len(exp_original_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params, _), exp_result_df) in enumerate(
    exp_original_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(data=exp_result_df, x=obj_var, y="utility", s=20, alpha=0.5, c="r")
    utility = exp_result_df["utility"].to_numpy()
    y_gt = exp_result_df[obj_var].to_numpy()
    a_1, a_0 = np.polyfit(y_gt, utility, 1)
    xs = np.linspace(y_gt.min(), y_gt.max(), 100)
    plt.plot(xs, a_1 * xs + a_0, "k--", lw=2)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"Actual {obj_var}", fontsize=10)
    plt.ylabel("Utility function", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/preference_learning_scatter.png", format="png", dpi=300, bbox_inches="tight"
)
plt.savefig("images/preference_learning_scatter.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/preference_learning_scatter.svg", format="svg", bbox_inches="tight")
plt.show()